# CodeT5-Architecture Pretraining on TLM's Own Corpus -- Kaggle
To run this notebook follow the following notebook URL




## Cell 1 -- Imports, Dataset Mounts, Device

In [ ]:
!pip install -q orjson transformers

import sys, os, shutil
from pathlib import Path


_SRC_DATASET         = "/kaggle/input/tlm-s-code" 
_CURRICULUM_DATASET  = "/kaggle/input/curriclum"
_PRECOMPUTED_DATASET = "/kaggle/input/curriculum-precomputed"

if not os.path.exists('/kaggle/working/src'):
    shutil.copytree(_SRC_DATASET, '/kaggle/working/src')
sys.path.insert(0, '/kaggle/working')

import math, random, json, pickle
from typing import Dict, List
import numpy as np

from src.model_config import ModelConfig
from src.tokenizer import TLM_Tokenizer
from src.data_loaders import (
    collate_fn,
    load_curriculum_config,
    tiers_for_epoch,
    max_epoch_from_curriculum,
    make_concat_dataset,
    curriculum_total_optimizer_steps,
    load_precomputed_tiers,
)
from src.training import make_scheduler

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from torch.utils.data import DataLoader
from transformers import T5Config, T5ForConditionalGeneration

TOKENIZER_PATH    = "/kaggle/working/src/tokenizer/tokenizer_tlm_small.json"
_CURRICULUM_DIR   = Path(_CURRICULUM_DATASET)
_PRECOMPUTED_ROOT = Path(_PRECOMPUTED_DATASET)
TIERS             = [1, 2, 3, 4, 5]
N_VARIANTS_TRAIN  = 5

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")
print("Running on: Kaggle")
print(f"Curriculum config dir: {_CURRICULUM_DIR}  ({'OK' if (_CURRICULUM_DIR / 'curriculum_config.json').exists() else 'MISSING'})")
print(f"Precomputed corruption root: {_PRECOMPUTED_ROOT}")
for split in ("train", "val"):
    status = "OK" if (_PRECOMPUTED_ROOT / split).exists() else "MISSING"
    print(f"  {status}  {split}/")

## Cell 2 -- Tokenizer + Special Tokens


In [ ]:
tlm_config = ModelConfig()   # only used for special_token_ids / vocab_size below
PAD_ID = tlm_config.special_token_ids["pad"]
BOS_ID = tlm_config.special_token_ids["bos"]
EOS_ID = tlm_config.special_token_ids["eos"]

tokenizer = TLM_Tokenizer(vocab_size=tlm_config.vocab_size)
tokenizer.load(TOKENIZER_PATH)
VOCAB_SIZE = tokenizer.tokenizer.get_vocab_size()
print(f"Tokenizer loaded from {TOKENIZER_PATH}")
print(f"  vocab_size = {VOCAB_SIZE}  |  pad={PAD_ID}  bos={BOS_ID}  eos={EOS_ID}")

## Cell 3 -- CodeT5-Small Architecture, Random Init, Resized Vocab

`T5Config.from_pretrained(...)` only fetches the architecture's `config.json`
(layer counts, d_model, d_ff, attention heads, relative-position buckets,
dropout) -- it does **not** download or load any model weights. Building
`T5ForConditionalGeneration(t5_config)` from that config object gives a
randomly-initialized model with CodeT5-small's real architecture. Vocab size,
pad/bos/eos ids are overridden to match this project's own tokenizer instead
of CodeT5's original ~32k-token vocab -- necessary to reuse the exact same
corpus/corruption pairs, but it also means the resulting parameter count will
be measurably below CodeT5-small's real 60M (a smaller vocab means a smaller
tied embedding/LM-head matrix) -- printed below, not assumed.

Needs `internet: true` in `kaggle_codet5.yml` -- this fetch goes over the
network the first time the kernel runs.

In [ ]:
t5_config = T5Config.from_pretrained("Salesforce/codet5-small")
t5_config.vocab_size = VOCAB_SIZE
t5_config.pad_token_id = PAD_ID
t5_config.eos_token_id = EOS_ID
t5_config.decoder_start_token_id = BOS_ID   # matches this project's dec_ids[0] == BOS convention

print("CodeT5-small architecture (vocab overridden to this project's tokenizer):")
print(f"  d_model={t5_config.d_model}  d_ff={t5_config.d_ff}  "
      f"num_layers={t5_config.num_layers}  num_heads={t5_config.num_heads}  "
      f"relative_attention_num_buckets={t5_config.relative_attention_num_buckets}")

model = T5ForConditionalGeneration(t5_config).to(device)  # random init -- NOT .from_pretrained on the model

# Gradient checkpointing -- trades compute for a large activation-memory cut, needed
# here since Kaggle's GPU OOM'd on this architecture at the TLM ladder's own batch
# size. use_cache=False is required alongside it: KV-caching and activation
# checkpointing are incompatible during training (caching keeps exactly the
# activations checkpointing is trying to discard); use_cache only matters for
# autoregressive generation anyway, not this labels-based training forward pass.
model.gradient_checkpointing_enable()
model.config.use_cache = False

n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel params: {n_params/1e6:.3f}M "
      f"(CodeT5-small's real vocab would give ~60M; smaller vocab here reduces the "
      f"tied embedding/LM-head matrix -- this is the actual capacity being tested, "
      f"report this number, not a bare '60M' claim)")
print("Gradient checkpointing: enabled (use_cache=False)")

## Cell 4 -- Curriculum Data (identical to the TLM ladder's own pretrain run)

Same precomputed SEARCH/REPLACE corruption pairs, same curriculum schedule,
same `collate_fn` -- this is the part that must stay byte-for-byte identical
across the architecture comparison. `NUM_WORKERS=2` -- the known-safe setting
on Kaggle GPU kernels (`NUM_WORKERS=0` serializes CPU corruption with GPU
compute; see `project_colab_pretrain_run` memory).

In [ ]:
BATCH_SIZE        = 8    # micro-batch actually sent to the GPU per forward/backward --
                          # reduced from the TLM ladder's 32 after Kaggle OOM'd on this
                          # architecture; see GRAD_ACCUM_STEPS below for how the
                          # effective batch size is kept at 32 anyway
GRAD_ACCUM_STEPS  = 4     # BATCH_SIZE * GRAD_ACCUM_STEPS = 32 -- same effective batch
                          # size as the TLM ladder's own pretrain runs, so this is still
                          # "the same pipeline," just spread over more forward/backward
                          # passes per optimizer step to fit in memory
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_WORKERS = 2
_PIN        = torch.cuda.is_available()
_collate    = lambda b: collate_fn(b, PAD_ID)

TRAIN_DS_BY_TIER = load_precomputed_tiers(_PRECOMPUTED_ROOT / "train", TIERS, N_VARIANTS_TRAIN)
VAL_DS_BY_TIER   = load_precomputed_tiers(_PRECOMPUTED_ROOT / "val",   TIERS, 1)

print("precomputed examples per tier:")
for t in TIERS:
    print(f"  tier{t}: train {len(TRAIN_DS_BY_TIER[t]):,}  |  val {len(VAL_DS_BY_TIER[t]):,}")

val_ds = make_concat_dataset(VAL_DS_BY_TIER, TIERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)

CURRICULUM_CONFIG = load_curriculum_config(_CURRICULUM_DIR / "curriculum_config.json")
EPOCHS = max_epoch_from_curriculum(CURRICULUM_CONFIG)
print(f"\nCurriculum: {EPOCHS} epochs  |  micro-batch {BATCH_SIZE}  x  "
      f"{GRAD_ACCUM_STEPS} accum steps  =  effective batch {EFFECTIVE_BATCH_SIZE}")
for ep in range(1, EPOCHS + 1):
    active = tiers_for_epoch(ep, CURRICULUM_CONFIG)
    n = sum(len(TRAIN_DS_BY_TIER[t]) for t in active if t in TRAIN_DS_BY_TIER)
    print(f"  epoch {ep:02d}: tiers {active} | train examples {n:,}")

## Cell 5 -- HF Forward/Loss Step

`T5ForConditionalGeneration` computes its own teacher-forced cross-entropy
loss internally when `labels` is passed. To match this project's own
`dec_ids = [BOS, ...body..., EOS]` convention exactly: pass `labels =
dec_ids[:, 1:]` (drop the leading BOS) with `decoder_start_token_id = BOS_ID`
already set on the config above -- HF's internal `_shift_right(labels)` then
reconstructs `decoder_input_ids = [BOS] + dec_ids[1:-1]`, identical to what
this project's own `dec_in = dec_ids[:, :-1]` / `dec_tgt = dec_ids[:, 1:]`
split does elsewhere. Padding positions in `labels` are set to `-100`, i.e.
HF's cross-entropy ignore-index, so the loss matches `compute_loss`'s own
pad-masking behavior. No `p_gen` copy-gate penalty here -- that mechanism is
TLM_S-specific (`CopyPointerHead`) and CodeT5's architecture has no
equivalent, which is itself part of what's being compared, not something to
paper over.

In [ ]:
# Mixed precision -- OFF by default. T5 is well-documented to be numerically
# unstable in fp16 (loss -> NaN): unlike most transformers, T5 doesn't scale
# attention logits by sqrt(d_k), which fp16's narrow dynamic range handles badly.
# HF's own guidance is bf16 or fp32 for T5, never fp16. Kaggle's typical T4 GPU
# has no bf16 tensor-core acceleration (Turing architecture predates it), so
# enabling bf16 here would only help MEMORY (halved activation dtype size), not
# speed, on that hardware. Left off by default -- gradient checkpointing +
# batch/accum tuning are the safe levers; only flip this on if still OOMing
# after those, and watch train_loss closely for NaN/inf if you do.
USE_AMP = False
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_available()
                               and torch.cuda.is_bf16_supported()) else None
if USE_AMP and AMP_DTYPE is None:
    print("WARNING: USE_AMP=True but this GPU has no bf16 support -- refusing to fall back "
          "to fp16 for a T5 model (known NaN-loss risk, see comment above). Running in fp32.")


def hf_step_batch(model, batch, pad_id, device, amp_dtype=None):
    enc_ids  = batch["enc_ids"].to(device)
    enc_attn = batch["enc_attn"].to(device)
    dec_ids  = batch["dec_ids"].to(device)

    labels = dec_ids[:, 1:].clone()
    labels[labels == pad_id] = -100

    if amp_dtype is not None:
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            out = model(input_ids=enc_ids, attention_mask=enc_attn, labels=labels)
    else:
        out = model(input_ids=enc_ids, attention_mask=enc_attn, labels=labels)
    return out.loss


@torch.no_grad()
def hf_validate(model, loader, pad_id, device, amp_dtype=None, log_every=200):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for i, batch in enumerate(loader):
        loss = hf_step_batch(model, batch, pad_id, device, amp_dtype=amp_dtype)
        total_loss += float(loss.detach())
        n_batches += 1
        if (i + 1) % log_every == 0:
            print(f"    [val] {i+1}/{len(loader)}", flush=True)
    return total_loss / max(n_batches, 1)


def hf_run_epoch(model, loader, optimizer, scheduler, pad_id, device,
                 grad_accum_steps=1, amp_dtype=None, grad_clip=1.0, log_every=200):
    """Gradient-accumulation-aware: only steps optimizer/scheduler every
    `grad_accum_steps` micro-batches, so BATCH_SIZE can be shrunk for memory
    without changing the effective batch size the LR schedule/optimizer see.
    Flushes one final partial-window step at epoch end if the number of
    micro-batches (after DataLoader's drop_last=True) isn't an exact multiple
    of grad_accum_steps, so no accumulated gradient is ever silently dropped."""
    model.train()
    total_loss, n_micro = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for i, batch in enumerate(loader):
        loss = hf_step_batch(model, batch, pad_id, device, amp_dtype=amp_dtype)
        (loss / grad_accum_steps).backward()
        total_loss += float(loss.detach())
        n_micro += 1

        if n_micro % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if (i + 1) % log_every == 0:
            print(f"    step {i+1}/{len(loader)}  loss={total_loss/n_micro:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}", flush=True)

    if n_micro % grad_accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / max(n_micro, 1)

## Cell 6 -- Run Config + Resume Support

Same hyperparameters as the TLM ladder's pretrain runs (`LR`, `WEIGHT_DECAY`,
`WARMUP_FRAC`, `PATIENCE`, `GRAD_CLIP`) -- kept identical on purpose, same
reasoning as `BATCH_SIZE` above.

**`/kaggle/working` is ephemeral** -- download `CKPT_DIR` (see
`kaggle_codet5.yml`'s `outputs.download_to`) before the session ends, or this
checkpoint history is lost when the kernel is cleaned up, same caveat as the
TLM ladder's own Kaggle notebook.

In [ ]:
RUN_NAME = "codet5_arch_pretrain"
RUN_MODE = "fresh"   # "fresh" (start new) or "resume" (continue this RUN_NAME's last.pt)
assert RUN_MODE in ("fresh", "resume"), f"RUN_MODE must be 'fresh' or 'resume', got {RUN_MODE!r}"

CKPT_DIR = Path("/kaggle/working") / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# /kaggle/input is always mounted read-only, including your own prior kernel's
# output added as an input dataset -- so it can only ever be a READ source (for
# resuming from a previous version), never a write target. The in-progress
# resume state for *this* run must be written under /kaggle/working instead.
LAST_CKPT_PATH = CKPT_DIR / "last.pt"
RESUME_PATH = Path("/kaggle/input/codet5-pretraining/codet5_arch_pretrain/last.pt")

LR           = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC  = 0.03
PATIENCE     = 4
GRAD_CLIP    = 1.0

optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# EFFECTIVE_BATCH_SIZE here, not BATCH_SIZE -- curriculum_total_optimizer_steps counts
# *optimizer* steps, and with gradient accumulation each optimizer step now spans
# GRAD_ACCUM_STEPS micro-batches, so the effective batch size is what actually
# determines how many optimizer steps (and therefore the warmup/cosine schedule) there are.
total_steps = curriculum_total_optimizer_steps(CURRICULUM_CONFIG, TRAIN_DS_BY_TIER, EFFECTIVE_BATCH_SIZE)
warmup      = int(WARMUP_FRAC * total_steps)
scheduler   = make_scheduler(optimizer, warmup_steps=warmup, total_steps=total_steps)

start_epoch = 1
best_val_loss = float("inf")
patience_ctr = 0
history = {"epoch": [], "train_loss": [], "val_loss": []}

if RUN_MODE == "resume":
    assert RESUME_PATH.exists(), f"RUN_MODE='resume' but no last.pt at {RESUME_PATH}"
    ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    patience_ctr = ckpt.get("patience_ctr", 0)
    history = ckpt.get("history", history)
    print(f"Resumed from {RESUME_PATH.name}: starting at epoch {start_epoch}, "
          f"best_val_loss={best_val_loss:.4f}")
else:
    assert not LAST_CKPT_PATH.exists(), (
        f"RUN_MODE='fresh' but {LAST_CKPT_PATH} already exists -- this would silently ignore "
        f"prior progress already saved under RUN_NAME={RUN_NAME!r}. Use RUN_MODE='resume' "
        f"to continue it, or pick a new RUN_NAME for a genuinely new run."
    )
    print(f"Fresh run: {RUN_NAME}  |  total_steps={total_steps}  warmup={warmup}")
print(f"checkpoints -> {CKPT_DIR}")

## Cell 7 -- Training Loop (curriculum, resumable, early stopping)

In [ ]:
print("\n" + "=" * 70)
print(f"STARTING CURRICULUM PRETRAINING -- epoch {start_epoch}..{EPOCHS}")
print("=" * 70)

for epoch in range(start_epoch, EPOCHS + 1):
    active_tiers = tiers_for_epoch(epoch, CURRICULUM_CONFIG)
    for t in active_tiers:
        TRAIN_DS_BY_TIER[t].set_variant(epoch - 1)   # cycle through the 5 precomputed variants
    train_ds = make_concat_dataset(TRAIN_DS_BY_TIER, active_tiers)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=_collate, num_workers=NUM_WORKERS,
                              pin_memory=_PIN, drop_last=True,
                              persistent_workers=NUM_WORKERS > 0)

    print(f"\n=== Epoch {epoch:02d}/{EPOCHS} | tiers {active_tiers} | "
          f"train samples {len(train_ds):,} | batches {len(train_loader)} ===", flush=True)
    train_loss = hf_run_epoch(model, train_loader, optimizer, scheduler, PAD_ID, device,
                              grad_accum_steps=GRAD_ACCUM_STEPS, amp_dtype=AMP_DTYPE, grad_clip=GRAD_CLIP)
    val_loss = hf_validate(model, val_loader, PAD_ID, device, amp_dtype=AMP_DTYPE)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"ppl={math.exp(min(val_loss, 20)):.4f}")

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        patience_ctr = 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": val_loss,
                    "t5_config": t5_config.to_dict()},
                   CKPT_DIR / f"{RUN_NAME}_epoch_{epoch:02d}_val{val_loss:.4f}.pt")
        print(f"  [best so far -> saved]")
    else:
        patience_ctr += 1
        print(f"  no val improvement ({patience_ctr}/{PATIENCE})")

    # Written to /kaggle/working (LAST_CKPT_PATH), never RESUME_PATH -- /kaggle/input
    # is read-only, so writing there raises "Read-only file system" (inline_container.cc:743).
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "epoch": epoch,
                "best_val_loss": best_val_loss, "patience_ctr": patience_ctr,
                "history": history}, LAST_CKPT_PATH)

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)")
        break

print("\n" + "=" * 70)
print("CURRICULUM PRETRAINING COMPLETE")
print(f"Best val loss : {best_val_loss:.4f}")
print("=" * 70)

## Cell 8 -- Save Training Curves

Same CSV + plot convention as the TLM ladder's own pretrain notebooks, for a
direct side-by-side comparison against the TLM-M pretrain curve. Remember:
download `CKPT_DIR` before the Kaggle session ends.

In [ ]:
import csv
import matplotlib.pyplot as plt

csv_path = CKPT_DIR / "training_curves.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(history.keys()))
    writer.writeheader()
    for i in range(len(history["epoch"])):
        writer.writerow({k: history[k][i] for k in history})
print(f"Saved training curves to {csv_path}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(history["epoch"], history["train_loss"], label="train loss", marker="o")
ax.plot(history["epoch"], history["val_loss"], label="val loss", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title(f"{RUN_NAME} ({n_params/1e6:.1f}M params) -- loss curves")
ax.legend()
plt.tight_layout()
plt.savefig(CKPT_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRemember: /kaggle/working is ephemeral -- download {CKPT_DIR} "
      f"(see kaggle_codet5.yml's outputs.download_to) before the session ends.")